# ISE 427 Tıpta Yapay Zeka - Proje Ödevi (Revize Edilmiş Özgün Modeller)

**Ders:** ISE 427 Tıpta Yapay Zeka  
**Konu:** Kalp Hastalığı Teşhisi için Standart ve Özgün Makine Öğrenmesi Modelleri

### Projenin Amacı
Bu notebook, Cleveland kalp hastalığı veri seti üzerinde **5 standart** (LR, SVM, DT, RF, KNN) ve proje isterlerine uygun olarak geliştirilmiş **5 özgün** algoritmanın eğitimini, optimizasyonunu ve karşılaştırmasını içerir.

### Geliştirilen Özgün Yaklaşımlar (Literatür Destekli)
1.  **KNN:** `Log-Karesel Mesafe Metriği` (Outlier'ların etkisini azaltmak için logaritmik ölçekleme).
2.  **SVM:** `Bessel-Laplacian Hibrit Çekirdek` (Tıbbi sinyallerdeki karmaşık örüntüleri yakalamak için).
3.  **Lojistik Regresyon:** `ISRU (Inverse Square Root Unit) Aktivasyonu` (Sigmoid'e alternatif, cebirsel olasılık dönüşümü).
4.  **Karar Ağacı:** `Hellinger Uzaklığı Kazancı` (Sınıf dağılımları arasındaki istatistiksel mesafeyi maksimize eder, dengesiz veride Gini'den daha iyidir).
5.  **Rastgele Orman:** `Dinamik Ağaç Seçimi (DTS)` (Her test örneği için, o örneğin uzaydaki komşularında en iyi performans gösteren ağaçları seçip oylatır. Basit çoğunluk veya ağırlıklı oylamadan çok daha akıllı bir sistemdir).

---

In [1]:
# --- 1. Kütüphaneler ve Veri Yükleme ---
import warnings
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial.distance import cdist

# Scikit-learn
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.neighbors import KNeighborsClassifier

# Standart Modeller
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

# Verileri Yükle (Preprocessing notebook'undan gelenler)
if Path("preprocess_pipeline.pkl").exists():
    preprocess_pipeline = joblib.load("preprocess_pipeline.pkl")
    X_train, X_test, y_train, y_test = joblib.load("train_test_data.pkl")
    print("Veriler ve Pipeline yüklendi.")
else:
    print("Lütfen önce Preprocessing notebook'unu çalıştırın!")

Veriler ve Pipeline yüklendi.


In [2]:
# --- 2. Değerlendirme Fonksiyonu ---
def evaluate_model(model, X_te, y_te, model_name):
    y_pred = model.predict(X_te)
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    
    return {
        "Algoritma": model_name,
        "Başarı Oranı (Acc)": accuracy_score(y_te, y_pred),
        "DN (True Neg)": tn,
        "YP (False Pos)": fp,
        "YN (False Neg)": fn,
        "DP (True Pos)": tp,
        "Duyarlılık (Recall)": recall_score(y_te, y_pred, zero_division=0),
        "Özgüllük (Spec)": tn / (tn + fp) if (tn+fp)>0 else 0,
        "Kesinlik (Prec)": precision_score(y_te, y_pred, zero_division=0),
        "F Ölçütü (F1)": f1_score(y_te, y_pred, zero_division=0)
    }

## 1: Özgün Algoritmaların Kodlanması

In [3]:
# --- ÖZGÜN 1: KNN için "Log-Karesel (Log-Quadratic)" Mesafe Metriği ---
# Literatür: Öklid mesafesi büyük farkların karesini alarak outlier'lara aşırı duyarlı hale gelir.
# Çözüm: Farkların logaritmasını alarak bu etkiyi yumuşatan, tıbbi veriye uygun "Robust" bir metrik.
# Formül: D(x,y) = Sum( log(1 + |xi - yi|^2) )

class LogQuadraticKNN(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=5):
        self.n_neighbors = n_neighbors
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        # Veri tiplerini garantiye al
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        return self

    def _log_quad_dist(self, a, b):
        # Özelleştirilmiş Mesafe Hesabı
        diff = a - b
        return np.sum(np.log1p(diff**2))

    def predict(self, X):
        X = np.array(X)
        preds = []
        # Her test örneği için
        for x_sample in X:
            # Tüm eğitim setine olan mesafeyi hesapla
            dists = np.array([self._log_quad_dist(x_sample, x_tr) for x_tr in self.X_train])
            # En yakın k komşuyu bul
            k_indices = np.argsort(dists)[:self.n_neighbors]
            k_labels = self.y_train[k_indices]
            # Oylama
            preds.append(np.bincount(k_labels).argmax())
        return np.array(preds)

In [4]:
# --- ÖZGÜN 2: SVM için "Bessel-Laplacian Hibrit" Çekirdek Fonksiyonu ---
# Literatür: RBF çekirdeği (Gaussian) tek bir sigma parametresi ile her yere aynı davranır.
# Çözüm: Laplacian (keskin tepeler için) ve Bessel (dalgalı yapılar için) fonksiyonlarını birleştiren hibrit yapı.
# Formül: K(x,y) = exp(-|x-y|/sigma) * (1 + |x-y|^2)^(-beta)

def bessel_laplacian_kernel(X, Y, sigma=1.0, beta=0.5):
    # Öklid mesafesi matrisi (Pairwise)
    dists = cdist(X, Y, metric='euclidean')
    
    # Laplacian Kısmı: exp(-d/sigma)
    K_lap = np.exp(-dists / sigma)
    
    # Bessel/Rational Kısmı: (1 + d^2)^-beta
    K_bes = np.power(1 + dists**2, -beta)
    
    return K_lap * K_bes

class HybridSVM(SVC):
    def __init__(self, C=1.0, sigma=1.0, beta=0.5):
        super().__init__(kernel='precomputed', C=C, probability=True)
        self.sigma = sigma
        self.beta = beta
        self.X_train = None

    def fit(self, X, y):
        self.X_train = np.array(X)
        # Gram Matrisini Hesapla
        K = bessel_laplacian_kernel(self.X_train, self.X_train, self.sigma, self.beta)
        return super().fit(K, np.array(y))

    def predict(self, X):
        X = np.array(X)
        K = bessel_laplacian_kernel(X, self.X_train, self.sigma, self.beta)
        return super().predict(K)

In [5]:
# --- ÖZGÜN 3: Lojistik Regresyon için "ISRU" Aktivasyon Fonksiyonu ---
# Literatür: Sigmoid fonksiyonu exp() işlemi nedeniyle hesaplama maliyeti yüksektir ve 0-1'e çok hızlı doyar (saturation).
# Çözüm: "Inverse Square Root Unit" (ISRU) tabanlı cebirsel olasılık fonksiyonu. Daha yumuşak geçiş sağlar.
# Formül: P(z) = 0.5 * (1 + z / sqrt(1 + alpha * z^2))

class LogisticISRU(BaseEstimator, ClassifierMixin):
    def __init__(self, alpha=1.0, learning_rate=0.01, epochs=1000):
        self.alpha = alpha
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None

    def _isru(self, z):
        # Cebirsel Olasılık Dönüşümü (0 ile 1 arasına map eder)
        return 0.5 * (1 + z / np.sqrt(1 + self.alpha * z**2))

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        # Basit Gradient Descent
        for _ in range(self.epochs):
            linear_model = np.dot(X, self.weights) + self.bias
            y_predicted = self._isru(linear_model)
            
            # Hata türevi (Basitleştirilmiş update kuralı)
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)
            
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
        return self

    def predict(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted = self._isru(linear_model)
        return [1 if i > 0.5 else 0 for i in y_predicted]

In [6]:
# --- ÖZGÜN 4: Karar Ağacı için "Hellinger Uzaklığı" Bilgi Kazancı ---
# Literatür: Gini ve Entropi, sınıf dengesizliğine karşı hassas değildir. 
# Çözüm: Hellinger mesafesi, iki olasılık dağılımı arasındaki farkı geometrik olarak ölçer ve ayrımı keskinleştirir.
# Formül: Gain = sqrt( sum( (sqrt(p_left) - sqrt(p_right))^2 ) )

class HellingerTree(BaseEstimator, ClassifierMixin):
    def __init__(self, max_depth=5, min_samples=5):
        self.max_depth = max_depth
        self.min_samples = min_samples
        self.tree = None

    def _hellinger_gain(self, left_y, right_y):
        # Hellinger Distance Hesabı
        def get_prob(y):
            if len(y) == 0: return 0
            return np.sum(y) / len(y) # Pozitif sınıf olasılığı

        p_L = get_prob(left_y)
        p_R = get_prob(right_y)
        
        # Hellinger Distance: (sqrt(p1)-sqrt(p2))^2 + (sqrt(1-p1)-sqrt(1-p2))^2
        dist = (np.sqrt(p_L) - np.sqrt(p_R))**2 + (np.sqrt(1-p_L) - np.sqrt(1-p_R))**2
        return np.sqrt(dist)

    def fit(self, X, y):
        self.tree = self._build_tree(np.array(X), np.array(y), depth=0)
        return self

    def _build_tree(self, X, y, depth):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        # Durma kriterleri
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples):
            return np.mean(y) # Yaprak değeri (olasılık)

        best_gain = -1
        best_split = None

        # En iyi bölümlemeyi bul
        for feat_idx in range(n_feats):
            thresholds = np.unique(X[:, feat_idx])
            for thr in thresholds:
                left_idx = X[:, feat_idx] <= thr
                right_idx = ~left_idx
                if np.sum(left_idx) == 0 or np.sum(right_idx) == 0: continue
                
                gain = self._hellinger_gain(y[left_idx], y[right_idx])
                if gain > best_gain:
                    best_gain = gain
                    best_split = (feat_idx, thr, X[left_idx], y[left_idx], X[right_idx], y[right_idx])
        
        if best_gain == -1:
            return np.mean(y)

        # Recursive olarak ağacı büyüt
        idx, thr, X_l, y_l, X_r, y_r = best_split
        return {
            'feat': idx, 'thr': thr,
            'left': self._build_tree(X_l, y_l, depth+1),
            'right': self._build_tree(X_r, y_r, depth+1)
        }

    def predict(self, X):
        return np.array([1 if self._predict_one(x, self.tree) >= 0.5 else 0 for x in np.array(X)])

    def _predict_one(self, x, node):
        if not isinstance(node, dict):
            return node
        if x[node['feat']] <= node['thr']:
            return self._predict_one(x, node['left'])
        else:
            return self._predict_one(x, node['right'])

In [7]:
# --- ÖZGÜN 5: Rastgele Orman için "Dinamik Ağaç Seçimi (Dynamic Tree Selection)" ---
# Literatür: Çoğunluk oyu (Majority Voting), tüm ağaçların her örnek için eşit derecede iyi olduğunu varsayar. 
# Çözüm: Test edilen örneğe (query instance) uzayda en yakın olan eğitim örneklerinde (Validation set üzerinde)
# hangi ağaçlar başarılı olmuşsa, sadece o "uzman" ağaçların oyunu dikkate alan dinamik yapı.
# Bu yöntem "Ağırlıklı" değil "Seçici (Selective)" bir topluluk (Ensemble) yöntemidir.

class DynamicSelectionRF(BaseEstimator, ClassifierMixin):
    def __init__(self, n_estimators=50, k_neighbors=7):
        self.n_estimators = n_estimators
        self.k_neighbors = k_neighbors
        self.forest = []
        self.X_val = None
        self.y_val = None
        self.knn = None

    def fit(self, X, y):
        # NumPy array dönüşümü (Index hatasını önlemek için)
        X = np.array(X)
        y = np.array(y)
        
        # Veriyi Eğitim ve Validasyon (Yetkinlik ölçümü için) olarak ikiye böl
        split_idx = int(len(X) * 0.7)
        X_tr, y_tr = X[:split_idx], y[:split_idx]
        self.X_val, self.y_val = X[split_idx:], y[split_idx:]
        
        # Ormanı Eğit
        for _ in range(self.n_estimators):
            # Bootstrap Sample
            indices = np.random.choice(len(X_tr), len(X_tr), replace=True)
            tree = DecisionTreeClassifier(max_depth=5, max_features='sqrt')
            tree.fit(X_tr[indices], y_tr[indices])
            self.forest.append(tree)
        
        # Komşuluk araması için KNN eğit (Validasyon seti üzerinde)
        self.knn = NearestNeighbors(n_neighbors=self.k_neighbors)
        self.knn.fit(self.X_val)
        return self

    def predict(self, X):
        X = np.array(X)
        final_preds = []
        
        # Her test örneği için Dinamik Seçim
        for i in range(len(X)):
            sample = X[i].reshape(1, -1)
            # 1. Test örneğinin validasyon setindeki en yakın komşularını bul
            _, neighbor_indices = self.knn.kneighbors(sample)
            neighbor_indices = neighbor_indices[0]
            
            # 2. Her ağacın bu komşular üzerindeki performansını ölç (Lokal Yetkinlik)
            competent_trees = []
            for tree in self.forest:
                val_preds = tree.predict(self.X_val[neighbor_indices])
                accuracy = np.mean(val_preds == self.y_val[neighbor_indices])
                if accuracy > 0.5: # Sadece %50'den iyi olan ağaçları seç
                    competent_trees.append(tree)
            
            # 3. Seçilen ağaçlarla oylama yap
            if not competent_trees: competent_trees = self.forest # Hiçbiri iyi değilse hepsini kullan
            
            votes = [tree.predict(sample)[0] for tree in competent_trees]
            final_preds.append(max(set(votes), key=votes.count)) # Çoğunluk oyu (seçilmişler arasında)
            
        return np.array(final_preds)

## 2: Model Eğitimi ve Karşılaştırma

In [8]:
# Modelleri Tanımla
models = [
    # Standart Modeller
    ("LR (Standart)", LogisticRegression(max_iter=1000)),
    ("SVM (Standart)", SVC(kernel='rbf', probability=True)),
    ("DT (Standart)", DecisionTreeClassifier(max_depth=5)),
    ("RF (Standart)", RandomForestClassifier(n_estimators=50, random_state=42)),
    ("KNN (Standart)", KNeighborsClassifier(n_neighbors=5)),
    
    # Özgün Modeller
    ("LR (Özgün: ISRU)", LogisticISRU(alpha=1.0)),
    ("SVM (Özgün: Bessel-Lap)", HybridSVM(C=1.0, sigma=2.0)),
    ("DT (Özgün: Hellinger)", HellingerTree(max_depth=5)),
    ("RF (Özgün: Dyn. Select)", DynamicSelectionRF(n_estimators=50)),
    ("KNN (Özgün: Log-Quad)", LogQuadraticKNN(n_neighbors=5))
]

results = []

print("Eğitim Başladı...\n")

for name, model in models:
    print(f"... {name} eğitiliyor.")
    
    # Pipeline ile birleştir (Ön işleme + Model)
    # Not: Ön işleme zaten dışarıda yapıldıysa direkt fit edebiliriz ama pipeline daha güvenlidir.
    # Burada X_train zaten transform edilmiş durumda mı? Kontrol edelim.
    # Eğer train_test_data.pkl ham veriyse pipeline kullanmalı.
    
    try:
        # Pipeline kullanarak ham veri ile eğit (Data Leakage önlemi)
        clf = Pipeline([('preprocessor', preprocess_pipeline), ('classifier', model)])
        clf.fit(X_train, y_train)
        
        # Test
        res = evaluate_model(clf, X_test, y_test, name)
        results.append(res)
    except Exception as e:
        print(f"Hata ({name}): {e}")

print("\nTüm modeller tamamlandı.")

Eğitim Başladı...

... LR (Standart) eğitiliyor.
... SVM (Standart) eğitiliyor.
... DT (Standart) eğitiliyor.
... RF (Standart) eğitiliyor.
... KNN (Standart) eğitiliyor.
... LR (Özgün: ISRU) eğitiliyor.
... SVM (Özgün: Bessel-Lap) eğitiliyor.
... DT (Özgün: Hellinger) eğitiliyor.
... RF (Özgün: Dyn. Select) eğitiliyor.
... KNN (Özgün: Log-Quad) eğitiliyor.

Tüm modeller tamamlandı.


In [9]:
# --- Sonuçları Tablo Haline Getir ---
df_results = pd.DataFrame(results)
cols = ["Algoritma", "Başarı Oranı (Acc)", "DN (True Neg)", "YP (False Pos)", 
        "YN (False Neg)", "DP (True Pos)", "Duyarlılık (Recall)", "Özgüllük (Spec)", 
        "Kesinlik (Prec)", "F Ölçütü (F1)"]
df_results = df_results[cols].sort_values(by="F Ölçütü (F1)", ascending=False)

print("\nPERFORMANS KARŞILAŞTIRMASI")
display(df_results)

# Sonuçları Kaydet
df_results.to_csv("Final_Proje_Sonuclari.csv", index=False)


PERFORMANS KARŞILAŞTIRMASI


,Algoritma,Başarı Oranı (Acc),DN (True Neg),YP (False Pos),YN (False Neg),DP (True Pos),Duyarlılık (Recall),Özgüllük (Spec),Kesinlik (Prec),F Ölçütü (F1)
9,KNN (Özgün: Log-Quad),0.934426,30,3,1,27,0.964286,0.909091,0.900000,0.931034
6,SVM (Özgün: Bessel-Lap),0.901639,28,5,1,27,0.964286,0.848485,0.843750,0.900000
3,RF (Standart),0.901639,29,4,2,26,0.928571,0.878788,0.866667,0.896552
4,KNN (Standart),0.901639,29,4,2,26,0.928571,0.878788,0.866667,0.896552
5,LR (Özgün: ISRU),0.885246,28,5,2,26,0.928571,0.848485,0.838710,0.881356
1,SVM (Standart),0.885246,28,5,2,26,0.928571,0.848485,0.838710,0.881356
0,LR (Standart),0.868852,28,5,3,25,0.892857,0.848485,0.833333,0.862069
8,RF (Özgün: Dyn. Select),0.868852,29,4,4,24,0.857143,0.878788,0.857143,0.857143
2,DT (Standart),0.737705,24,9,7,21,0.750000,0.727273,0.700000,0.724138
7,DT (Özgün: Hellinger),0.540984,31,2,26,2,0.071429,0.939394,0.500000,0.125000
